# Results Analysis — Clinical QA System
**LJMU MSc ML\&AI — Amit Choudhary (1187247)**

Comprehensive analysis of all experimental conditions and metric comparisons for the thesis.

In [ ]:
import sys
sys.path.insert(0, "..")
import json, math
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import seaborn as sns
from src.config import settings
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 130
plt.rcParams["font.size"] = 11
print("Setup complete.")

## 1. Load All Condition Results

In [ ]:
def load_condition(cond):
    for name in [f"e2e_{cond}_results.jsonl", f"medqa_predictions_{cond}.jsonl"]:
        p = settings.data_processed_dir / name
        if p.exists():
            return [json.loads(l) for l in open(p) if l.strip()]
    return []

conditions = ["base_llm", "rag_only", "rag_with_hallucination_filter"]
label_map  = {"base_llm":"Base LLM", "rag_only":"RAG Only",
              "rag_with_hallucination_filter":"RAG + Halluc Filter
(Proposed)"}

cdata = {c: load_condition(c) for c in conditions}
available = {c: d for c, d in cdata.items() if d}

for c, d in available.items():
    if d:
        acc   = np.mean([r.get("correct", False) for r in d])
        halluc = np.mean([r.get("hallucination_rate", 0) for r in d])
        t     = np.mean([r.get("response_time_s", 0) for r in d])
        print(f"{c:40s}  acc={acc:.4f}  halluc={halluc:.4f}  time={t:.2f}s")
    else:
        print(f"{c}: no data — run: python scripts/run_pipeline.py --condition {c}")

## 2. Main Comparison Table (Thesis Table 4.1)

In [ ]:
rows = []
for c in conditions:
    d = cdata.get(c, [])
    if not d:
        rows.append({"Condition": label_map[c].replace("
"," "), "N":0,
                     "Accuracy":None,"Halluc Rate":None,"Avg Time (s)":None,"P95 Time (s)":None})
        continue
    accs  = [r.get("correct",False) for r in d]
    hs    = [r.get("hallucination_rate",0) for r in d]
    ts    = [r.get("response_time_s",0) for r in d]
    rows.append({"Condition": label_map[c].replace("
"," "),
                 "N": len(d),
                 "Accuracy":  round(np.mean(accs), 4),
                 "Halluc Rate":round(np.mean(hs),  4),
                 "Avg Time (s)":round(np.mean(ts), 2),
                 "P95 Time (s)":round(np.percentile(ts, 95), 2) if ts else None})

df = pd.DataFrame(rows)
print("
Table 4.1 — Experimental Conditions Comparison")
print(df.to_string(index=False))
df.to_csv("../docs/table_4_1_comparison.csv", index=False)
print("
Saved to docs/table_4_1_comparison.csv")

## 3. Hallucination Reduction Analysis

In [ ]:
base_d = cdata.get("base_llm", [])
prop_d = cdata.get("rag_with_hallucination_filter", [])

if base_d and prop_d:
    base_h = np.mean([r.get("hallucination_rate", 0) for r in base_d])
    prop_h = np.mean([r.get("hallucination_rate", 0) for r in prop_d])
    reduction = (base_h - prop_h) / base_h * 100 if base_h > 0 else 0.0
    target_met = reduction >= 50.0
    print(f"Base LLM hallucination rate:     {base_h:.4f}")
    print(f"Proposed system hallucination:   {prop_h:.4f}")
    print(f"Reduction:                       {reduction:.1f}%")
    print(f"Target (>=50%):                  {chr(9989) if target_met else chr(10060)} {'MET' if target_met else 'NOT MET'}")
else:
    print("Run all conditions first: python scripts/run_pipeline.py --condition all")

## 4. Visualisations

In [ ]:
avail = {c: d for c, d in cdata.items() if d}
if len(avail) >= 2:
    lbls   = [label_map[c] for c in avail]
    accs   = [np.mean([r.get("correct",False) for r in d]) for d in avail.values()]
    halluc = [np.mean([r.get("hallucination_rate",0) for r in d]) for d in avail.values()]
    times  = [np.mean([r.get("response_time_s",0) for r in d]) for d in avail.values()]
    p95s   = [np.percentile([r.get("response_time_s",0) for r in d], 95) for d in avail.values()]
    colors = ["#d73027","#fee090","#1a9850"][:len(avail)]

    fig = plt.figure(figsize=(16, 10))
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

    # Accuracy
    ax0 = fig.add_subplot(gs[0, 0])
    bars = ax0.bar(range(len(lbls)), accs, color=colors, alpha=0.85, edgecolor="black")
    ax0.axhline(0.75, color="red", linestyle="--", linewidth=1.5, label="Target 75%")
    ax0.set_xticks(range(len(lbls))); ax0.set_xticklabels(lbls, fontsize=9)
    ax0.set_ylabel("MedQA Accuracy"); ax0.set_title("(a) MedQA Accuracy", fontweight="bold")
    ax0.set_ylim(0, 1.0); ax0.legend(fontsize=9)
    for i, a in enumerate(accs): ax0.text(i, a+0.01, f"{a:.3f}", ha="center", fontsize=9, fontweight="bold")

    # Hallucination rate
    ax1 = fig.add_subplot(gs[0, 1])
    bars1 = ax1.bar(range(len(lbls)), halluc, color=colors, alpha=0.85, edgecolor="black")
    ax1.set_xticks(range(len(lbls))); ax1.set_xticklabels(lbls, fontsize=9)
    ax1.set_ylabel("Hallucination Rate"); ax1.set_title("(b) Hallucination Rate", fontweight="bold")
    for i, h in enumerate(halluc): ax1.text(i, h+0.005, f"{h:.3f}", ha="center", fontsize=9, fontweight="bold")

    # Response time
    ax2 = fig.add_subplot(gs[1, 0])
    x = np.arange(len(lbls)); w = 0.35
    ax2.bar(x - w/2, times, w, label="Mean", color="steelblue", alpha=0.8)
    ax2.bar(x + w/2, p95s,  w, label="P95",  color="coral",     alpha=0.8)
    ax2.axhline(3.0, color="red", linestyle="--", linewidth=1.5, label="SLA 3s")
    ax2.set_xticks(x); ax2.set_xticklabels(lbls, fontsize=9)
    ax2.set_ylabel("Time (s)"); ax2.set_title("(c) Response Time", fontweight="bold")
    ax2.legend(fontsize=9)

    # Hallucination distribution
    ax3 = fig.add_subplot(gs[1, 1])
    for i, (c, d) in enumerate(avail.items()):
        hs = [r.get("hallucination_rate",0) for r in d]
        ax3.hist(hs, bins=20, alpha=0.6, label=label_map[c].replace("
"," "), color=colors[i])
    ax3.set_xlabel("Hallucination Rate"); ax3.set_ylabel("Count")
    ax3.set_title("(d) Hallucination Distribution", fontweight="bold")
    ax3.legend(fontsize=8)

    Path("../docs").mkdir(exist_ok=True)
    plt.savefig("../docs/main_results_figure.png", bbox_inches="tight", dpi=150)
    plt.show()
    print("Saved → docs/main_results_figure.png")
else:
    print("Need at least 2 conditions of data. Run: python scripts/run_pipeline.py --condition all")

## 5. Per-question Accuracy Analysis

In [ ]:
best = cdata.get("rag_with_hallucination_filter", [])
if best:
    correct_times  = [r["response_time_s"] for r in best if r.get("correct")]
    wrong_times    = [r["response_time_s"] for r in best if not r.get("correct")]
    correct_halluc = [r["hallucination_rate"] for r in best if r.get("correct")]
    wrong_halluc   = [r["hallucination_rate"] for r in best if not r.get("correct")]
    print(f"Correct predictions: {len(correct_times)}")
    print(f"  Avg halluc rate: {np.mean(correct_halluc):.4f}" if correct_halluc else "")
    print(f"  Avg time: {np.mean(correct_times):.2f}s" if correct_times else "")
    print(f"Wrong predictions: {len(wrong_times)}")
    print(f"  Avg halluc rate: {np.mean(wrong_halluc):.4f}" if wrong_halluc else "")
    print(f"  Avg time: {np.mean(wrong_times):.2f}s" if wrong_times else "")
    if correct_halluc and wrong_halluc:
        print(f"
Hallucination rate is {'higher' if np.mean(wrong_halluc) > np.mean(correct_halluc) else 'lower'} on wrong predictions — expected pattern.")
else:
    print("No proposed-system results yet.")

## 6. MLflow Experiment Summary

In [ ]:
import mlflow
mlflow.set_tracking_uri(settings.mlflow_tracking_uri)
client = mlflow.tracking.MlflowClient()
try:
    exp = mlflow.get_experiment_by_name(settings.mlflow_experiment_name)
    if not exp:
        print("No MLflow experiment found. Run: python scripts/setup_mlflow.py")
    else:
        runs = client.search_runs([exp.experiment_id], order_by=["attributes.start_time DESC"], max_results=10)
        print(f"Experiment: {settings.mlflow_experiment_name}")
        print(f"Total runs: {len(runs)}")
        print()
        for r in runs:
            name = r.data.tags.get("mlflow.runName", r.info.run_id[:8])
            acc  = r.data.metrics.get("accuracy", float("nan"))
            hall = r.data.metrics.get("hallucination_rate", float("nan"))
            print(f"  {name:<40s} acc={acc:.4f}  halluc={hall:.4f}")
except Exception as e:
    print(f"MLflow error: {e}")